# Projet d'Actuariat IMAFA 2026-2027
## Logiciel d'Assurance de Risque Météorologique
**Enseignant :** Marius AKRE  
**Étudiant :** Amiel Metier  

---

### Objectifs du notebook :
1. Modéliser l'activité d'un commerçant face au risque de pluie ($CA_t^{pl}$, $R_t^{pl}$).
2. Déterminer le seuil de rentabilité pluviométrique $pl^*$.
3. Calculer la tarification annuelle d'assurance (approche prospective).
4. Émettre un devis client imprimable.
5. Réaliser une analyse rétrospective sur l'exemple du sujet (Nice, 1998) avec visualisation graphique.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Inclusion du dossier src
sys.path.insert(0, os.path.abspath("../src"))

from assurance_meteo.model import ProfilClient, simulation_activite_journaliere
from assurance_meteo.weather import recuperer_pluviometrie_annuelle, recuperer_historique_pluviometrie
from assurance_meteo.pricing import calculer_prime_assurance, indemnites_annuelles_historiques
from assurance_meteo.devis import generer_devis
from assurance_meteo.retrospective import analyser_annee_retrospective, tracer_analyse_retrospective

## 1. Modélisation économique du commerçant

D'après le sujet (Section I), le commerçant présente :
- Un chiffre d'affaires maximal journalier : $CA$
- Des coûts fixes incompressibles : $Cf$ (avec $CA > Cf$)
- Un niveau de pluie journalier pivot : $\overline{pl}$

Le facteur d'atténuation du chiffre d'affaires est donné par :
$$f_t^{pl} = \frac{\overline{pl} - pl_t}{\overline{pl}} \quad (0 < pl_t < \overline{pl})$$

Le résultat journalier est à perte ($R_t^{pl} < 0$) dès que la pluie dépasse le seuil critique :
$$pl^* = \overline{pl}\left(1 - \frac{Cf}{CA}\right)$$

In [ ]:
client = ProfilClient(
    nom="Commerçant Niçois",
    ville="Nice",
    date_souscription="1998-01-01",
    ca_max=500.0,       # CA journalier max
    couts_fixes=200.0,  # Coûts fixes journaliers
    pluie_pivot=10.0    # Pluie pivot en mm
)

print(f"Client : {client.nom} ({client.ville})")
print(f"Seuil de rentabilité (pl*) : {client.seuil_perte:.2f} mm/jour")
print(f"-> Au-delà de {client.seuil_perte:.2f} mm de pluie, le résultat journalier devient négatif.")

## 2. Données météorologiques historiques (Nice)

Récupération des données réelles de précipitations pour Nice en 1998.

In [ ]:
serie_1998 = recuperer_pluviometrie_annuelle("Nice", 1998, cache_dir="../data")
print(f"Nombre de jours relevés : {len(serie_1998)}")
print(f"Cumul annuel de pluie   : {serie_1998.sum():.1f} mm")
print(f"Précipitation journalière max : {serie_1998.max():.1f} mm")

## 3. Tarification prospective (Section II)

Le contrat d'assurance garantit au client un résultat positif ($R_t \ge 0$) pour chaque jour de l'année :
$$I_t = \max(0, -R_t^{pl})$$

La prime pure correspond à l'espérance mathématique des pertes annuelles $\mathbb{E}[L]$, à laquelle s'ajoutent les chargements de sécurité et de gestion.

In [ ]:
df_1998 = simulation_activite_journaliere(client, serie_1998)

pertes_reelles = df_1998["indemnite_assurance"].sum()
r_nass = df_1998["resultat_non_assure"].sum()
r_ass = df_1998["resultat_assure_brut"].sum()
nb_jours_sinistres = df_1998["jour_a_perte"].sum()

print(f"Nombre de jours indemnisés en 1998 : {nb_jours_sinistres} jours")
print(f"Résultat annuel non assuré (R_nass) : {r_nass:,.2f} €")
print(f"Pertes réelles prises en charge (L) : {pertes_reelles:,.2f} €")
print(f"Résultat annuel garanti (R_ass)     : {r_ass:,.2f} €")

## 4. Devis commercial (Section III)

Édition du devis avec les paramètres du client, les garanties et le tarif.

In [ ]:
tarif_devis = {
    "prime_pure": 2500.0,
    "chargement_securite": 300.0,
    "chargement_frais": 200.0,
    "prime_totale": 3000.0,
    "nb_jours_sinistre_moyen": 15,
    "nb_annees_observees": 15
}
devis = generer_devis(client, tarif_devis)
print(devis)

## 5. Analyse rétrospective — Exemple Nice 1998 (Section IV)

On teste l'exemple du sujet :
- Client niçois ($i$) en 1998 ($a$)
- Niveau de pluie pivot : $\overline{pl} = 10$ mm
- Prime a priori : $P = 3\,000$ €
- Vérification de la rentabilité de l'assurance a posteriori.

In [ ]:
bilan_1998 = analyser_annee_retrospective(
    client=client,
    annee=1998,
    prime_imposee=3000.0,
    cache_dir="../data"
)

print("=== BILAN RÉTROSPECTIF 1998 ===")
print(f"Résultat non assuré (R_nass)    : {bilan_1998['resultat_non_assure']:,.2f} €")
print(f"Prime payée au début (P)        : {bilan_1998['prime_a_priori']:,.2f} €")
print(f"Pertes réelles évitées          : {bilan_1998['pertes_reelles']:,.2f} €")
print(f"Résultat net assuré (R_ass - P) : {bilan_1998['resultat_assure_net']:,.2f} €")
print(f"Gain net pour le client         : {bilan_1998['gain_net_assurance']:+,.2f} €")
print(f"Assurance favorable au client   : {'OUI' if bilan_1998['assurance_favorable'] else 'NON'}")

### Représentation graphique comparative

Illustration graphique de la trajectoire du résultat (avec vs sans assurance) et des seuils pluviométriques.

In [ ]:
fig, axes = tracer_analyse_retrospective(
    bilan_1998,
    fichier_sauvegarde="../report/figures/fig_retrospective_nice_1998.png"
)
plt.show()